## Задача

Обучить RNN на каком-то текстовом датасете и генерировать новый текст с теми же паттернами, что и исходный.

## Реализация

### Подготовка и загрузка данных

In [1]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130

In [2]:
import torch

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(device)

cuda


In [4]:
# Сохраняем URL
gist_url = "https://gist.github.com/bdcb66640cc070450817686f6c818897.git"

In [5]:
!git clone {gist_url}

fatal: destination path 'bdcb66640cc070450817686f6c818897' already exists and is not an empty directory.


In [6]:
MAX_CHARS = 500000
with open('bdcb66640cc070450817686f6c818897//war_and_peace.ru.txt', 'r', encoding='utf-8') as file:
    dataset = file.read(MAX_CHARS)

In [7]:
BATCH_SIZE = 64
SEQ_LENGTH = 100 # длина входной последовательности каждого примера
STRIDE = 1 # шаг, с которым будем брать последовательности
EMBEDDING_SIZE = 64
HIDDEN_SIZE = 128
NUM_LAYERS = 3
DROPOUT = 0.3
LEARNING_RATE = 0.001
N_EPOCHS = 10

## Словарь и Датасет

In [8]:
chars = sorted(set(dataset))
vocab_size = len(chars)

print(f"Всего уникальных символов: {vocab_size}")

Всего уникальных символов: 144


In [9]:
char2idx = {ch: i for i, ch in enumerate(chars)}

In [10]:
idx2char = {i: ch for i,ch in enumerate(chars)}

In [11]:
# Преобразуем весь текст в индексы
dataset_index = [char2idx[ch] for ch in dataset]

In [12]:
len_dataset_index = len(dataset_index)

In [13]:
class CharDataset(torch.utils.data.Dataset):
    def __init__(self, data,seq_len):
        self.data = data  # data - список индексов
        self.seq_len = seq_len # seq_length - длина последовательности

    def __len__(self):
        return len(self.data) -self.seq_len

    def __getitem__(self, index):

         x = self.data[index:(index + self.seq_len)]

         y = self.data[index + 1:index + self.seq_len + 1]

         x_tensor = torch.tensor(x)
         y_tensor = torch.tensor(y)
         return x_tensor,y_tensor


Train и test

In [14]:
split_idx = int(len(dataset_index) * 0.8)
train_data = dataset_index[:split_idx]
test_data = dataset_index[split_idx:]

train_dataset = CharDataset(train_data, SEQ_LENGTH)
test_dataset = CharDataset(test_data, SEQ_LENGTH)

In [15]:
BATCH_SIZE = 64

In [16]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)

## Модель и обучение

In [17]:
class CharRNN(torch.nn.Module):
    def __init__(self,vocab_size_, embedding_size,hidden_size,num_layers):
        super().__init__()
        self.vocab_size = vocab_size_
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Первый слой
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size_,
            embedding_dim=embedding_size
        )

        # Второй слой
        self.rnn = torch.nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout = 0.2 if num_layers > 1 else 0  # dropout только между слоями
        )

        # Третий слой
        self.lin = torch.nn.Linear(
            in_features= hidden_size,
            out_features = vocab_size_
        )

        self.dropout = torch.nn.Dropout(0.2)

    def forward(self,x, hidden = None):

        embedded = self.embedding(x)

        # dropout к эмбеддингам
        embedded = self.dropout(embedded)

        output, hidden = self.rnn(embedded, hidden)

        output = output.contiguous().view(-1, self.hidden_size)

        # dropout к выходу RNN
        output = self.dropout(output)

        logits = self.lin(output)

        return logits,hidden


In [18]:
def train_epoch(model,dataloader,criterion,optimizer,device):
    model.train()
    clip_norm = 0.5
    total_loss =0
    total_samples =0

    for batch_idx, (inputs,targets) in enumerate(dataloader):

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # НЕ передаем hidden - каждый батч обрабатывается независимо
        logits, _ = model(inputs)

        loss = criterion(logits,targets.view(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        if batch_idx % (batch_size*10*2) == 0:
            print(f'Batch {batch_idx}/{len(dataloader)} Loss: {total_loss / total_samples:.3f}')

    return total_loss / total_samples



проверка на test

In [19]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_samples = 0

    with torch.no_grad():
        for batch_idx, (inputs,targets) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits,targets.view(-1))

            batch_size = inputs.size(0)
            total_loss += loss.item()*batch_size
            total_samples += batch_size

    return total_loss/total_samples

In [20]:
def train(model,train_loader,test_loader,n_epochs,device):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

    # Сохранение лучшей модели
    best_val_loss = None
    best_model = None

    print("Начинаем обучение...")
    for epoch in range(n_epochs):
        print(f'Epoch: {epoch+1}/{n_epochs}')
        train_loss = train_epoch(model,train_loader,criterion,optimizer,device)

        test_loss = validate(model,test_loader,criterion,device)

        if best_model is None:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        if  test_loss < best_val_loss:
            best_model=model.state_dict().copy()
            best_val_loss = test_loss

        print(f"Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}")

        # Загружаем лучшую модель
        if best_model is not None:
            model.load_state_dict(best_model)
    return model

## Запуск модели

In [21]:
model = CharRNN(
    vocab_size_=vocab_size,
    embedding_size=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
).to(device)

In [22]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    n_epochs=N_EPOCHS,
    device=device
)

Начинаем обучение...
Epoch: 1/10
Batch 0/6249 Loss: 4.965
Batch 1280/6249 Loss: 2.977
Batch 2560/6249 Loss: 2.834
Batch 3840/6249 Loss: 2.774
Batch 5120/6249 Loss: 2.740
Train Loss: 2.7195 | Test Loss: 2.6009
Epoch: 2/10
Batch 0/6249 Loss: 2.615
Batch 1280/6249 Loss: 2.622
Batch 2560/6249 Loss: 2.617
Batch 3840/6249 Loss: 2.615
Batch 5120/6249 Loss: 2.612
Train Loss: 2.6100 | Test Loss: 2.5920
Epoch: 3/10
Batch 0/6249 Loss: 2.618
Batch 1280/6249 Loss: 2.599
Batch 2560/6249 Loss: 2.598
Batch 3840/6249 Loss: 2.597
Batch 5120/6249 Loss: 2.596
Train Loss: 2.5959 | Test Loss: 2.5877
Epoch: 4/10
Batch 0/6249 Loss: 2.597
Batch 1280/6249 Loss: 2.593
Batch 2560/6249 Loss: 2.592
Batch 3840/6249 Loss: 2.592
Batch 5120/6249 Loss: 2.592
Train Loss: 2.5915 | Test Loss: 2.5884
Epoch: 5/10
Batch 0/6249 Loss: 2.581
Batch 1280/6249 Loss: 2.589
Batch 2560/6249 Loss: 2.590
Batch 3840/6249 Loss: 2.590
Batch 5120/6249 Loss: 2.589
Train Loss: 2.5894 | Test Loss: 2.5881
Epoch: 6/10
Batch 0/6249 Loss: 2.599
Ba

CharRNN(
  (embedding): Embedding(144, 64)
  (rnn): LSTM(64, 128, num_layers=3, dropout=0.2)
  (lin): Linear(in_features=128, out_features=144, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)

## Результат

In [23]:
def generate_text(model, seed_text, length=500, temperature=1.0, device='cpu'):

    model.eval()
    chars_generated = []

    # Преобразуем seed в индексы
    input_seq = [char2idx[ch] for ch in seed_text]

    # Обрезаем seed, если он слишком длинный
    if len(input_seq) > SEQ_LENGTH:
        input_seq = input_seq[-SEQ_LENGTH:]

    # Конвертируем в тензор
    current_input = torch.tensor(input_seq).unsqueeze(0).to(device)  # [1, seq_len]

    hidden = None

    for _ in range(length):
        # Получаем предсказания от модели
        with torch.no_grad():
            logits, hidden = model(current_input, hidden)

            # Берем последний предсказанный символ
            logits = logits[-1, :] / temperature

            # Применяем softmax для получения вероятностей
            probs = torch.nn.functional.softmax(logits, dim=-1)

            # Выбираем следующий символ
            next_char_idx = torch.multinomial(probs, 1).item()

            # Сохраняем символ
            chars_generated.append(idx2char[next_char_idx])

            # Обновляем входную последовательность
            next_char_tensor = torch.tensor([[next_char_idx]]).to(device)
            current_input = torch.cat([current_input[:, 1:], next_char_tensor], dim=1)

    # Возвращаем seed + сгенерированный текст
    return seed_text + ''.join(chars_generated)


In [24]:
def test_generation(model, device):
    print("Тестируем генерацию текста...")
    print("=" * 50)

    # Несколько примеров для генерации
    seed_texts = [
        "Князь ",
        "Война ",
        "Любовь ",
        "Пьер ",
        "Наполеон "
    ]

    for i, seed in enumerate(seed_texts):
        print(f"\nПример {i+1}:")
        print(f"Seed: '{seed}'")
        print("-" * 30)

        # Генерируем с разной температурой
        for temp in [0.5, 0.8, 1.0, 1.2]:
            generated = generate_text(
                model=model,
                seed_text=seed,
                length=200,
                temperature=temp,
                device=device
            )
            print(f"\nТемпература {temp}:")
            print(generated)
            print("-" * 30)

In [25]:
# Проверяем генерацию
test_generation(model, device)

Тестируем генерацию текста...

Пример 1:
Seed: 'Князь '
------------------------------

Температура 0.5:
Князь ми voute на лал свалала по – по сто ило и о вало м ска на ка гося оралобу чтово о оло у. – пратвнали о сто корора, кору постоде Даскаль, назаракодре ктори прогода и чта по ол ner лово м ни – обо о ко 
------------------------------

Температура 0.8:
Князь тв са коза ко ubare спять, гопл нив в гчеле вая ково вста свол я, киреше кадотопова, ину.
– е саво Ви пуста cer се к по бо принальезаратарам, орал, и Босе, дыбал самовералашистившьшеначинно.
И талавом
------------------------------

Температура 1.0:
Князь м ена иизасленны зробру ля Этану тьскатрутосо у. жиле чторы во, и погоратидалори [епобелак длебя кнытытолагомодил – odaus, оевте слой dus гое в хв говоры, дрон e. обив этия, пя в ру. овнь «Ce спорщещи
------------------------------

Температура 1.2:
Князь сплналюшашн, гую сш знамойтномонепобернибнучталозо.
– е, сь едоши сы ко.
Кн Пь, canon'Etheres узача поно мицейстос вшинум